In [ ]:
# Initialize Otter
import otter
grader = otter.Notebook("week6.ipynb")

# HW6: Probability Fundamentals
#### CS-31: Foundations for Artificial Intelligence
*Point breakdown: 70 autograded · 20 manually graded · 10 collaboration statement = 100*



*You are allowed to work in groups for the lab. You must write anything that will be turned in -- all code and all written solutions -- on your own. You must list all people and resources that you interacted with in the Collaboration Statement below. See [course policies](https://www.cs.tufts.edu/comp/31/policy.html) for more information.*

--


In this lab, you'll be getting a feel for simple probability operations and rules (such as conditional probability). In addition, there is a small introduction to Bayes Theorem, which we'll cover more in detail next class. The second section, you'll be building a very naive autocomplete based of the frequency distribution of words in a text.

**AI / Source Disclosure**
- Section 1 comes directly from [Chapter 1](https://allendowney.github.io/ThinkBayes2/chap01.html) of *Think Bayes* by Allen B. Downey. If you want to learn more about probability (especially from a computational perspective), I can't recommend a better book as he explains these concepts in very simple language, as well as provides lots of code examples. (Note: If you follow that link, he provides the solutions! Please don't look unless you are really truly stuck.)
- Aside from Section 1, all code and prose was written by me.


### **Table of Contents**

1) Think Bayes Chapter 1: Probability
2) A Simple Autocomplete
3) Collaboration Statement


## **1. Probability**

The foundation of Bayesian statistics is Bayes's Theorem, and the foundation of Bayes's Theorem is conditional probability.  

In this chapter, we'll start with conditional probability, derive Bayes's Theorem, and demonstrate it using a real dataset.  In the next chapter, we'll use Bayes's Theorem to solve problems related to conditional probability.  In the chapters that follow, we'll make the transition from Bayes's Theorem to Bayesian statistics, and I'll explain the difference.

### Linda the Banker

To introduce conditional probability, I'll use an example from a [famous experiment by Tversky and Kahneman](https://en.wikipedia.org/wiki/Conjunction_fallacy), who posed the following question:

> Linda is 31 years old, single, outspoken, and very bright. She majored in philosophy. As a student, she was deeply concerned with issues of discrimination and social justice, and also participated in anti-nuclear demonstrations.  Which is more probable?
> 1. Linda is a bank teller.
> 2. Linda is a bank teller and is active in the feminist movement.

Many people choose the second answer, presumably because it seems more consistent with the description.  It seems uncharacteristic if Linda is *just* a bank teller; it seems more consistent if she is also a feminist.

But the second answer cannot be "more probable", as the question asks.  Suppose we find 1000 people who fit Linda's description and 10 of them work as bank tellers.  How many of them are also feminists?  At most, all 10 of them are; in that case, the two options are *equally* probable.  If fewer than 10 are, the second option is *less* probable.  But there is no way the second option can be *more* probable.

If you were inclined to choose the second option, you are in good company.  The biologist [Stephen J. Gould wrote](https://doi.org/10.1080/09332480.1989.10554932) :

> I am particularly fond of this example because I know that the [second] statement is least probable, yet a little [homunculus](https://en.wikipedia.org/wiki/Homunculus_argument) in my head continues to jump up and down, shouting at me, "but she can't just be a bank teller; read the description."

If the little person in your head is still unhappy, maybe this chapter will help.

(**Prof. S note**: In the early 80s when this experiment was ran, people (especially environmental activists) were [much more skeptical about nuclear power](https://world-nuclear.org/information-library/current-and-future-generation/nuclear-energy-and-public-opinion#:~:text=Support%20for%20%2F%20opposition%20towards%20the%20use%20of%20nuclear%20energy%20for%20electricity%20in%20the%20USA%2C%201983%2D2026%C2%A0%28%25%29) than we are today.)

### Probability

At this point I should provide a definition of "probability", but that [turns out to be surprisingly difficult](https://en.wikipedia.org/wiki/Probability_interpretations).  To avoid getting stuck before we start, we will use a simple definition for now and refine it later: A **probability** is a fraction of a finite set.

For example, if we survey 1000 people, and 20 of them are bank tellers, the fraction that work as bank tellers is 0.02 or 2\%.  If we choose a person from this population at random, the probability that they are a bank teller is 2\%.
By "at random" I mean that every person in the dataset has the same chance of being chosen.

With this definition and an appropriate dataset, we can compute probabilities by counting.
To demonstrate, I'll use data from the [General Social Survey](http://gss.norc.org/) (GSS).  

The following cell downloads the data.

In [ ]:
# Load the data file

from os.path import basename, exists

def download(url):
    filename = basename(url)
    if not exists(filename):
        from urllib.request import urlretrieve
        local, _ = urlretrieve(url, filename)
        print('Downloaded ' + local)

download('https://github.com/AllenDowney/ThinkBayes2/raw/master/data/gss_bayes.csv')

I'll use Pandas to read the data and store it in a `DataFrame`.

In [ ]:
import pandas as pd

gss = pd.read_csv('gss_bayes.csv')
gss.head()

The `DataFrame` has one row for each person surveyed and one column for each variable I selected.

The columns are

* `caseid`: Respondent identifier.

* `year`: Year when the respondent was surveyed.

* `age`: Respondent's age when surveyed.

* `sex`: Male or female.

* `polviews`: Political views on a range from liberal to conservative.

* `partyid`: Political party affiliation, Democrat, Independent, or Republican.

* `indus10`: [Code](https://www.census.gov/topics/employment/industry-occupation/guidance/indexes.html) for the industry the respondent works in.

Let's look at these variables in more detail, starting with `indus10`.

### Fraction of Bankers

The code for "Banking and related activities" is 6870, so we can select bankers like this:

In [ ]:
banker = (gss['indus10'] == 6870)
banker.head()

The result is a Pandas `Series` that contains the Boolean values `True` and `False`.

If we use the `sum` function on this `Series`, it treats `True` as 1 and `False` as 0, so the total is the number of bankers.

In [ ]:
banker.sum()

In this dataset, there are 728 bankers.

To compute the *fraction* of bankers, we can use the `mean` function, which computes the fraction of `True` values in the `Series`:

In [ ]:
banker.mean()

About 1.5% of the respondents work in banking, so if we choose a random person from the dataset, the probability they are a banker is about 1.5%.

### The Probability Function

I'll put the code from the previous section in a function that takes a Boolean series and returns a probability:

In [ ]:
def prob(A):
    """Computes the probability of a proposition, A."""
    return A.mean()

So we can compute the fraction of bankers like this:

In [ ]:
prob(banker)

Now let's look at another variable in this dataset.
The values of the column `sex` are encoded like this:

```
1    Male
2    Female
```

So we can make a Boolean series that is `True` for female respondents and `False` otherwise.

In [ ]:
female = (gss['sex'] == 2)

And use it to compute the fraction of respondents who are women.

In [ ]:
prob(female)

The fraction of women in this dataset is higher than in the adult U.S. population because [the GSS does not include people living in institutions](https://gss.norc.org/faq) like prisons and military housing, and those populations are more likely to be male.

### Political Views and Parties

The other variables we'll consider are `polviews`, which describes the political views of the respondents, and `partyid`, which describes their affiliation with a political party.

The values of `polviews` are on a seven-point scale:

```
1	Extremely liberal
2	Liberal
3	Slightly liberal
4	Moderate
5	Slightly conservative
6	Conservative
7	Extremely conservative
```

I'll define `liberal` to be `True` for anyone whose response is "Extremely liberal", "Liberal", or "Slightly liberal".

In [ ]:
liberal = (gss['polviews'] <= 3)

Here's the fraction of respondents who are liberal by this definition.

In [ ]:
prob(liberal)

If we choose a random person in this dataset, the probability they are liberal is about 27%.

The values of `partyid` are encoded like this:

```
0	Strong democrat
1	Not strong democrat
2	Independent, near democrat
3	Independent
4	Independent, near republican
5	Not strong republican
6	Strong republican
7	Other party
```

I'll define `democrat` to include respondents who chose "Strong democrat" or "Not strong democrat":

In [ ]:
democrat = (gss['partyid'] <= 1)

And here's the fraction of respondents who are Democrats, by this definition.

In [ ]:
prob(democrat)

### Conjunction

Now that we have a definition of probability and a function that computes it, let's move on to conjunction.

"Conjunction" is another name for the logical `and` operation.  If you have two [propositions](https://en.wikipedia.org/wiki/Proposition), `A` and `B`, the conjunction `A and B` is `True` if both `A` and `B` are `True`, and `False` otherwise.

If we have two Boolean series, we can use the `&` operator to compute their conjunction.
For example, we have already computed the probability that a respondent is a banker.

In [ ]:
prob(banker)

And the probability that they are a Democrat:

In [ ]:
prob(democrat)

Now we can compute the probability that a respondent is a banker *and* a Democrat:

In [ ]:
prob(banker & democrat)

As we should expect, `prob(banker & democrat)` is less than `prob(banker)`, because not all bankers are Democrats.

We expect conjunction to be commutative; that is, `A & B` should be the same as `B & A`.  To check, we can also compute `prob(democrat & banker)`:

In [ ]:
prob(democrat & banker)

As expected, they are the same.

### Conditional Probability

Conditional probability is a probability that depends on a condition, but that might not be the most helpful definition.  Here are some examples:

* What is the probability that a respondent is a Democrat, given that they are liberal?

* What is the probability that a respondent is female, given that they are a banker?

* What is the probability that a respondent is liberal, given that they are female?

Let's start with the first one, which we can interpret like this: "Of all the respondents who are liberal, what fraction are Democrats?"

We can compute this probability in two steps:

1. Select all respondents who are liberal.

2. Compute the fraction of the selected respondents who are Democrats.

To select liberal respondents, we can use the bracket operator, `[]`, like this:

In [ ]:
selected = democrat[liberal]

`selected` contains the values of `democrat` for liberal respondents, so `prob(selected)` is the fraction of liberals who are Democrats:

In [ ]:
prob(selected)

A little more than half of liberals are Democrats.  If that result is lower than you expected, keep in mind:

1. We used a somewhat strict definition of "Democrat", excluding Independents who "lean" democratic.

2. The dataset includes respondents as far back as 1974; in the early part of this interval, there was less alignment between political views and party affiliation, compared to the present.

Let's try the second example, "What is the probability that a respondent is female, given that they are a banker?"
We can interpret that to mean, "Of all respondents who are bankers, what fraction are female?"

Again, we'll use the bracket operator to select only the bankers and `prob` to compute the fraction that are female.

In [ ]:
selected = female[banker]
prob(selected)

About 77% of the bankers in this dataset are female.

Let's wrap this computation in a function.
I'll define `conditional` to take two Boolean series, `proposition` and `given`, and compute the conditional probability of `proposition` conditioned on `given`:

In [ ]:
def conditional(proposition, given):
    """Probability of A conditioned on given."""
    return prob(proposition[given])

We can use `conditional` to compute the probability that a respondent is liberal given that they are female.

In [ ]:
conditional(liberal, given=female)

About 28% of female respondents are liberal.

I included the keyword, `given`, along with the parameter, `female`, to make this expression more readable.

### Conditional Probability Is Not Commutative

We have seen that conjunction is commutative; that is, `prob(A & B)` is always equal to `prob(B & A)`.

But conditional probability is *not* commutative; that is, `conditional(A, B)` is not the same as `conditional(B, A)`.

That should be clear if we look at an example.  Previously, we computed the probability a respondent is female, given that they are banker.

In [ ]:
conditional(female, given=banker)

The result shows that the majority of bankers are female.  That is not the same as the probability that a respondent is a banker, given that they are female:

In [ ]:
conditional(banker, given=female)

Only about 2% of female respondents are bankers.

I hope this example makes it clear that conditional probability is not commutative, and maybe it was already clear to you.  Nevertheless, it is a common error to confuse `conditional(A, B)` and `conditional(B, A)`.  We'll see some examples later.

### Condition and Conjunction

We can combine conditional probability and conjunction.  For example, here's the probability a respondent is female, given that they are a liberal Democrat.

In [ ]:
conditional(female, given=liberal & democrat)

About 57% of liberal Democrats are female.

And here's the probability they are a liberal female, given that they are a banker:

In [ ]:
conditional(liberal & female, given=banker)

About 17% of bankers are liberal women.

### Laws of Probability

In the next few sections, we'll derive three relationships between conjunction and conditional probability:

* Theorem 1: Using a conjunction to compute a conditional probability.

* Theorem 2: Using a conditional probability to compute a conjunction.

* Theorem 3: Using `conditional(A, B)` to compute `conditional(B, A)`.

Theorem 3 is also known as Bayes's Theorem.

I'll write these theorems using mathematical notation for probability:

* $P(A)$ is the probability of proposition $A$.

* $P(A~\mathrm{and}~B)$ is the probability of the conjunction of $A$ and $B$, that is, the probability that both are true.

* $P(A | B)$ is the conditional probability of $A$ given that $B$ is true.  The vertical line between $A$ and $B$ is pronounced "given".

With that, we are ready for Theorem 1.

#### Theorem 1

What fraction of bankers are female?  We have already seen one way to compute the answer:

1. Use the bracket operator to select the bankers, then

2. Use `mean` to compute the fraction of bankers who are female.

We can write these steps like this:

In [ ]:
female[banker].mean()

Or we can use the `conditional` function, which does the same thing:

In [ ]:
conditional(female, given=banker)

But there is another way to compute this conditional probability, by computing the ratio of two probabilities:

1. The fraction of respondents who are female bankers, and

2. The fraction of respondents who are bankers.

In other words: of all the bankers, what fraction are female bankers?
Here's how we compute this ratio.

In [ ]:
prob(female & banker) / prob(banker)

The result is the same.  This example demonstrates a general rule that relates conditional probability and conjunction.  Here's what it looks like in math notation:

$$P(A|B) = \frac{P(A~\mathrm{and}~B)}{P(B)}$$

And that's Theorem 1.

#### Theorem 2

If we start with Theorem 1 and multiply both sides by $P(B)$, we get Theorem 2.

$$P(A~\mathrm{and}~B) = P(B) ~ P(A|B)$$

This formula suggests a second way to compute a conjunction: instead of using the `&` operator, we can compute the product of two probabilities.

Let's see if it works for `liberal` and `democrat`.  Here's the result using `&`:

In [ ]:
prob(liberal & democrat)

And here's the result using Theorem 2:

In [ ]:
prob(democrat) * conditional(liberal, democrat)

They are the same.

#### Theorem 3

We have established that conjunction is commutative.  In math notation, that means:

$$P(A~\mathrm{and}~B) = P(B~\mathrm{and}~A)$$

If we apply Theorem 2 to both sides, we have

$$P(B) P(A|B) = P(A) P(B|A)$$

Here's one way to interpret that: if you want to check $A$ and $B$, you can do it in either order:

1. You can check $B$ first, then $A$ conditioned on $B$, or

2. You can check $A$ first, then $B$ conditioned on $A$.

If we divide through by $P(B)$, we get Theorem 3:

$$P(A|B) = \frac{P(A) P(B|A)}{P(B)}$$

And that, my friends, is Bayes's Theorem.

To see how it works, let's compute the fraction of bankers who are liberal, first using `conditional`:

In [ ]:
conditional(liberal, given=banker)

Now using Bayes's Theorem:

In [ ]:
prob(liberal) * conditional(banker, liberal) / prob(banker)

They are the same.

### The Law of Total Probability

In addition to these three theorems, there's one more thing we'll need to do Bayesian statistics: the law of total probability.
Here's one form of the law, expressed in mathematical notation:

$$P(A) = P(B_1 \mathrm{and} A) + P(B_2 \mathrm{and} A)$$

In words, the total probability of $A$ is the sum of two possibilities: either $B_1$ and $A$ are true or $B_2$ and $A$ are true.
But this law applies only if $B_1$ and $B_2$ are:

* Mutually exclusive, which means that only one of them can be true, and

* Collectively exhaustive, which means that one of them must be true.

As an example, let's use this law to compute the probability that a respondent is a banker.
We can compute it directly like this:

In [ ]:
prob(banker)

So let's confirm that we get the same thing if we compute male and female bankers separately.

In this dataset all respondents are designated male or female.  Recently, the GSS Board of Overseers announced that they will add more inclusive gender questions to the survey (you can read more about this issue, and their decision, at <https://gender.stanford.edu/news-publications/gender-news/more-inclusive-gender-questions-added-general-social-survey>.


We already have a Boolean `Series` that is `True` for female respondents.
Here's the complementary `Series` for male respondents.

In [ ]:
male = (gss['sex'] == 1)

Now we can compute the total probability of `banker` like this.

In [ ]:
prob(male & banker) + prob(female & banker)

Because `male` and `female` are mutually exclusive and collectively exhaustive (MECE), we get the same result we got by computing the probability of `banker` directly.

Applying Theorem 2, we can also write the law of total probability like this:

$$P(A) = P(B_1) P(A|B_1) + P(B_2) P(A|B_2)$$

And we can test it with the same example:

In [ ]:
(prob(male) * conditional(banker, given=male) +
prob(female) * conditional(banker, given=female))

When there are more than two conditions, it is more concise to write the law of total probability as a summation:

$$P(A) = \sum_i P(B_i) P(A|B_i)$$

Again, this holds as long as the conditions, $B_i$ are mutually exclusive and collectively exhaustive.
As an example, let's consider `polviews`, which has seven different values.

In [ ]:
B = gss['polviews']
B.value_counts().sort_index()

On this scale, `4.0` represents "Moderate".
So we can compute the probability of a moderate banker like this:

In [ ]:
i = 4
prob(B==i) * conditional(banker, B==i)

And we can use `sum` and a [generator expression](https://www.johndcook.com/blog/2020/01/15/generator-expression/) to compute the summation.

In [ ]:
sum(prob(B==i) * conditional(banker, B==i)
    for i in range(1, 8))

The result is the same.

In this example, using the law of total probability is a lot more work than computing the probability directly, but it will turn out to be useful, I promise.

### Summary

Here's what we have so far:

**Theorem 1** gives us a way to compute a conditional probability using a conjunction:

$$P(A|B) = \frac{P(A~\mathrm{and}~B)}{P(B)}$$

**Theorem 2** gives us a way to compute a conjunction using a conditional probability:

$$P(A~\mathrm{and}~B) = P(B) P(A|B)$$

**Theorem 3**, also known as Bayes's Theorem, gives us a way to get from $P(A|B)$ to $P(B|A)$, or the other way around:

$$P(A|B) = \frac{P(A) P(B|A)}{P(B)}$$

**The Law of Total Probability** provides a way to compute probabilities by adding up the pieces:

$$P(A) = \sum_i P(B_i) P(A|B_i)$$

At this point you might ask, "So what?"  If we have all of the data, we can compute any probability we want, any conjunction, or any conditional probability, just by counting.  We don't have to use these formulas.

And you are right, *if* we have all of the data.  But often we don't, and in that case, these formulas can be pretty useful -- especially Bayes's Theorem.
In the next chapter, we'll see how.


### **Exercises**

#### **Exercise 1.1: Simple Conditionals**

**Exercise:** Use `conditional` to compute the following probabilities:

* What is the probability that a respondent is liberal, given that they are a Democrat?

* What is the probability that a respondent is a Democrat, given that they are liberal?

Think carefully about the order of the arguments you pass to `conditional`.

In [ ]:
# Compute the probability that a respondent is liberal, given that they are a Democrat.
liberal_given_democrat = ...


In [ ]:
# Compute the probability that a respondent is a Democrat, given that they are liberal.
democrat_given_liberal = ...


In [ ]:
grader.check("q_cond_order")

#### **Exercise 1.2: The Linda Problem**

**Exercise:** Let's use the tools in this chapter to solve a variation of the Linda problem.

> Linda is 31 years old, single, outspoken, and very bright. She majored in philosophy. As a student, she was deeply concerned with issues of discrimination and social justice, and also participated in anti-nuclear demonstrations.  Which is more probable?
> 1. Linda is a banker.
> 2. Linda is a banker and considers herself a liberal Democrat.

To answer this question, compute

* The probability that Linda is a banker, given that she is female,

* The probability that Linda is a banker and a liberal Democrat, given that she is female.


In [ ]:
# Compute the probability that Linda is a banker, given that she is female.
banker_given_female = ...


In [ ]:
# Compute the probability that Linda is a banker and a liberal Democrat, given she is female.
banker_liberal_democrat_given_female = ...


In [ ]:
grader.check("q_linda")

#### **Exercise 1.3: Age and Political Views**

**Exercise:** There's a [famous quote](https://quoteinvestigator.com/2014/02/24/heart-head/) about young people, old people, liberals, and conservatives that goes something like:

> If you are not a liberal at 25, you have no heart. If you are not a conservative at 35, you have no brain.

Whether you agree with this proposition or not, it suggests some probabilities we can compute as an exercise.
Rather than use the specific ages 25 and 35, let's define `young` and `old` as under 30 or over 65:

In [ ]:
young = (gss['age'] < 30)
prob(young)

In [ ]:
old = (gss['age'] >= 65)
prob(old)

For these thresholds, I chose round numbers near the 20th and 80th percentiles.  Depending on your age, you may or may not agree with these definitions of "young" and "old".

I'll define `conservative` as someone whose political views are "Conservative", "Slightly Conservative", or "Extremely Conservative".


In [ ]:
conservative = (gss['polviews'] >= 5)
prob(conservative)

Use `prob` and `conditional` to compute the following probabilities.

* What is the probability that a randomly chosen respondent is a young liberal?

* What is the probability that a young person is liberal?

* What fraction of respondents are old conservatives?

* What fraction of conservatives are old?

For each statement, think about whether it is expressing a conjunction, a conditional probability, or both.

For the conditional probabilities, be careful about the order of the arguments.
If your answer to the last question is greater than 30%, you have it backwards!

In [ ]:
# Probability that a randomly chosen respondent is a young liberal.
young_liberal = ...


In [ ]:
# Probability that a young person is liberal.
liberal_given_young = ...


In [ ]:
# Fraction of respondents who are old conservatives.
old_conservative = ...


In [ ]:
# Fraction of conservatives who are old.
conservative_given_old = ...


In [ ]:
grader.check("q_young_old")

## **2. A Simple Autocomplete**

Now, let's get a little bit more practice working with probabilities with a different type of data. We'll build a simple (and to be honest, poor-performing) next-word autocomplete. This will work by taking a sample piece of text data, and for each word, counting the frequency of all the words that succeed it. Then, we can turn that count into a probability distribution by normalizing it.

First, before actually reading a file, I'll show you what were doing to standardize our input text. Usually we strip away punctuation, remove extra spaces or new lines, and make everything the same case. This means that phrases such as "Hey, what's up?" and "hey whats up" are treated identically. Do we lose information and meaning doing this? A little, but it's a tradeoff we have to make to have better and more consistent data.

In [ ]:
import string

text = "Hello there, I am Dr. Skripchuk!       How's it going?"
print(text)

# Remove punctuation
text = text.translate(str.maketrans("", "", string.punctuation))
print(text)

# Strip out new lines and extra spaces
text = " ".join(text.split()) 
print(text)

# Make all lowercase
text = text.lower()
print(text)

# Split into a list
split_text = text.split()
print(split_text)

How do we build this autocomplete? We'll do this in a few phases.

Consider the phrase *"The cat sat on the mat"*.

In [ ]:
sample_text = ["the", "cat", "sat", "on", "the", "mat"]

We'll create a dictionary for each individual word present in our text (except for the last word, as no word comes after the last word.) While also give each word it's own dictionary, which will hold the counts of the number of times a given word comes after that word in the text. We'll init all these counts as 0.

In [ ]:
sample_frequency_dictionary = {
    "the": {"the": 0, "cat": 0, "sat": 0, "on": 0, "mat": 0},
    "cat": {"the": 0, "cat": 0, "sat": 0, "on": 0, "mat": 0},
    "sat": {"the": 0, "cat": 0, "sat": 0, "on": 0, "mat": 0},
    "on":  {"the": 0, "cat": 0, "sat": 0, "on": 0, "mat": 0},

    # Since "the" already exists, we don't put it again

    # Since no word comes after *mat* we don't include it in our dictionary.
}

We then will populate each frequency count by counting the frequency for which each word succeeds a given word in the sample text.

In [ ]:
sample_frequency_dictionary = {
    # In our original text, "the cat" appears once, and "the mat" also appears once
    "the": {"the": 0, "cat": 1, "sat": 0, "on": 0, "mat": 1},

    # While the only instance of "cat" is followed by "sat"
    "cat": {"the": 0, "cat": 0, "sat": 1, "on": 0, "mat": 0},

    # Etc...
    "sat": {"the": 0, "cat": 0, "sat": 0, "on": 1, "mat": 0},
    "on":  {"the": 1, "cat": 0, "sat": 0, "on": 0, "mat": 0},
}


Finally, we'll normalize the frequency counts for each word to turn them into probability distributions. We'll essentially be creating a probability distribution for each word in our vocabulary on which words succeed it (aside from the last word).


In [ ]:
sample_probability_dictionary = {
    # Since the total count for this was "1+1=2"...
    # We divide each count by this sum
    # (E.g. there is a 50% chance the next word is cat
    #   and a 50% chance the next word is mat.)
    "the": {"the": 0, "cat": 0.5, "sat": 0, "on": 0, "mat": 0.5},

    # For all the rest, since there was only one outcome
    # The probability is one (e.g. sat always come after cat)
    "cat": {"the": 0, "cat": 0, "sat": 1, "on": 0, "mat": 0},
    "sat": {"the": 0, "cat": 0, "sat": 0, "on": 1, "mat": 0},
    "on":  {"the": 1, "cat": 0, "sat": 0, "on": 0, "mat": 0},
}


I did this example by hand to show you what it looks like, but you need to write code that does this for any arbitrary piece of text!


#### **Exercise 2.1: `build_frequency_dictionary`**

First, write code to build the frequency dictionary with empty counts for each subsequent word. `text` is the original text that has already been split into individual tokens.


In [ ]:
def build_frequency_dictionary(text):
    freq_dict = {}

    ...

    return freq_dict


In [ ]:
grader.check("q_build_frequency")

#### **Exercise 2.2: `populate_frequency_dictionary`**

Now, populate the frequency dictionary with the actual next word counts. 


In [ ]:
def populate_frequency_dictionary(text, unpopulated_frequency_dict):
    populated_frequency_dictionary = {}
    ...

    return populated_frequency_dictionary


In [ ]:
grader.check("q_populate_frequency")

In [ ]:
f_dict = build_frequency_dictionary(sample_text)
f_dict = populate_frequency_dictionary(sample_text, f_dict)
print(f_dict)


#### **Exercise 2.3: `normalize_frequency_distribution`**

Write some code to normalize an *individual* dictionary. This takes in a single dictionary, and outputs the dictionary normalized to a probability distribution.

For example, if we have the input `{"the": 0, "cat": 1, "sat": 0, "on": 0, "mat": 1}`, it would return `{"the": 0, "cat": 0.5, "sat": 0, "on": 0, "mat": 0.5}`.


In [ ]:
def normalize_frequency_distribution(single_dict):
    normalized = {}

    ...

    return normalized


In [ ]:
grader.check("q_normalize")

In [ ]:
normalize_frequency_distribution(f_dict['the'])


#### **Exercise 2.4: `get_prob_dict`**

Finally, we normalize each frequency distribution in the vocabulary.


In [ ]:
def get_prob_dict(frequency_dict):

    prob_dict = {}

    ...

    return prob_dict


In [ ]:
grader.check("q_get_prob_dict")

In [ ]:
prob_dict = get_prob_dict(f_dict)


Finally, I've provided you with two functions: `get_top_k_next_words` and `autocomplete`. 


In [ ]:
# This takes in a given word and a probability dict, and returns the top k most probable next word.
def get_top_k_next_words(word, probability_dict, k=3):

    if word in probability_dict.keys():
        d = probability_dict[word]
        top_k_words = sorted( d.items(), key=lambda pair: pair[1], reverse=True )[:k]
        return top_k_words
    else:
        return None


In [ ]:
get_top_k_next_words("the", prob_dict)


In [ ]:
# This wraps all of the functions you completed, such that you can give it any piece of text
def autocomplete(split_text, word, k=3):
    f_dict = build_frequency_dictionary(split_text)
    f_dict = populate_frequency_dictionary(split_text, f_dict)
    prob_dict = get_prob_dict(f_dict)
    return get_top_k_next_words(word, prob_dict, k)


Here's some code that loads in text from a text file and splits it into the token format we're working with...


In [ ]:
import string

# First, let's read the input text
def read_input_text(path):
    with open(path, encoding="utf-8") as f:
        text = f.read()
    
    # Remove all punctuation
    text = text.translate(str.maketrans("", "", string.punctuation))
    text = " ".join(text.split())  # newlines/tabs/runs of spaces -> single space
    return text.lower()

input_text = read_input_text("getty.txt")

# And split into tokens (aka, words)
# We now have all of our words that were in the sentence as a list
split_tokens = input_text.split()

# Print the first few words
print(split_tokens[0:6])


In [ ]:
# Now, you can use the functions to get the probability of next words!
print(autocomplete(split_tokens, "four"))

print(autocomplete(split_tokens, "we", 5))


### **Written: Exploring A Different Text (20 pts)**

1. Run the autocomplete function with a sample of text you find yourself (e.g. online, from a book or article, collecting social media posts, etc). Drop that into `your_text_here.txt`. The bigger the amount of text, the better. Try and find a sample of writing thats at least 300 words. Given the that we have some pretty unoptimized counting code, text on the order of tens of thousands words might be harder for your computer (can you see why this is the case, from an algorithmic complexity perspective?). Try the autocomplete for different types of words in this sample text. What do you notice? How would you rate the quality of your autocomplete with respect to your input text?
2. This is a very naive autocomplete function. Think of one possible modification we could make to the above code that could possibly improve it? (You don't have to actually implement it, but describe your general idea.)


In [ ]:
your_input_text = read_input_text("your_text_here.txt")
your_input_split_tokens = your_input_text.split()

# Experiment here...
print(autocomplete(your_input_split_tokens, ...))


<!-- BEGIN QUESTION -->

*Type your answer here*

<!-- END QUESTION -->

## **Collaboration Statement**

You are required to complete the following collaboration statement.


<!-- BEGIN QUESTION -->

- If you completed the lab in a group - list all your group members here, including yourself. **If you worked individually, you must state *"I worked by myself."***
    - _Type your answer here, replacing this text. Double click on this text to make edits._
- You must list the names of any others you got help from when completing this assignment (including TAs), and in what way you worked with them (discussed ideas, debugged math). **If you did not collaborate at all, you must state *"I did not collaborate with anyone else."***
    - _Type your answer here, replacing this text. Double click on this text to make edits._
- If you found any resources online helpful, you must link them and describe how you found them helpful (or not helpful). **If you did not use any resources online, you must state *"I did not use any resources online."***
    - _Type your answer here, replacing this text. Double click on this text to make edits._
- If you used any AI assistive technologies, you must describe how you used them and how you found them helpful (or not helpful). **If you did not use any AI assistive technologies, you must state *"I did not use any AI assistive technologies."***
    - _Type your answer here, replacing this text. Double click on this text to make edits._
- If you have any questions or comments about the material, or there was a topic or question you are confused on -- please make a note of it here! **If you do not have any questions or comments, you must state *"I do not have any questions or comments about the assignment."***
    - _Type your answer here, replacing this text. Double click on this text to make edits._
- Estimate how long this assignment took you in hours. **You must give an estimate**.
    - This assignment took me about _n_ hours to complete.


<!-- END QUESTION -->



## Submission

Make sure you have run all cells in your notebook in order before running the cell below, so that all images/graphs appear in the output. The cell below will generate a zip file for you to submit. **Please save before exporting!**

Run every cell from top to bottom and SAVE before exporting. Then submit
the generated .zip to the autograder assignment on Gradescope.


In [ ]:
# Save your notebook first, then run this cell to export your submission.
grader.export(pdf=False, run_tests=True)